# Prepare MILK10k dataset for VLM training

Convert the MILK10k training dump (clinical + dermoscopic pair per lesion) into canonical JSONL.

**Possible external replication**, not part of the main PAD-UFES-20 study. Use it only if labels, clinical photographs, and metadata are compatible; exclude overlapping cases and report results separately.

- **Source:** `data/datasets/MILK10K/` — `MILK10k_Training_{GroundTruth,Metadata,Supplement}.csv` + `MILK10k_Training_Input/IL_*/ISIC_*.jpg`
- **Output:** `data/processed/milk10k/clinical_context/`
- **Splits:** none on disk (training dump only) — 80/10/10 grouped by `lesion_id`, stratified by class, so both images of a lesion stay in the same split
- **One sample per image** (clinical close-up and dermoscopic), with `Image type` in the prompt
- **Labels:** one-hot 11-way mapped to canonical `code` only (`NV`→`NEV`, `AKIEC`→`ACK`, `SCCKA`→`SCC`, plus `BEN_OTH` / `INF` / `MAL_OTH`); gold is `{"code": "<CODE>"}`
- **Metadata:** age, sex, skin-tone class, site, diagnosis confirmation
- Source `diagnosis_full` is used only to map the sample onto that closed code (do not leak subtype text into the prompt or as a generated target)


## 1. Setup


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError(f"Could not find repo root (src/) from {Path.cwd()}")

sys.path.insert(0, str(ROOT / "src"))

from vlm_ft.data.canonical import milk10k_image_rel, milk10k_prompt_and_label
from vlm_ft.data.prepare import (
    count_existing_images,
    load_milk10k,
    preview_processed,
    print_source_eda,
    rel_to_root,
    require_source,
    rows_to_samples,
    split_by_group,
    write_processed_dataset,
)

CURRENT_SOURCES = "PAD, ISIC18, HC, Derm1M, MILK10K"

MILK_ROOT = ROOT / "data/datasets/MILK10K"
OUT_DIR = ROOT / "data/processed/milk10k/clinical_context"
require_source(MILK_ROOT, expected=CURRENT_SOURCES)


## 2. Load


In [ ]:
df = load_milk10k(MILK_ROOT)
print(df.shape)
print(df["image_type"].value_counts())
df.head()


## 3. EDA


In [ ]:
ok, total = count_existing_images(df, "image_rel", MILK_ROOT, limit=200)
print(f"image existence (first 200): {ok}/{total}")
print("images per lesion:\n", df.groupby("lesion_id").size().value_counts().to_string())

splits = split_by_group(df, group_col="lesion_id", label_col="code")
eda = pd.concat(splits.values(), ignore_index=True)
print_source_eda(
    eda,
    label_col="code",
    metadata_cols=["age_approx", "sex", "skin_tone_class", "site", "image_type", "diagnosis_confirm_type", "malignancy"],
    extra={"image_type": eda["image_type"].value_counts(), "task_code": eda["task_code"].value_counts()},
)


## 4. Convert to canonical JSONL


In [ ]:
MILK_ROOT_REL = rel_to_root(MILK_ROOT, ROOT)
samples = {
    split: rows_to_samples(
        frame,
        MILK_ROOT,
        prompt_and_label=milk10k_prompt_and_label,
        image_rel=milk10k_image_rel,
    )
    for split, frame in splits.items()
}
write_processed_dataset(
    name="milk10k/clinical_context",
    out_dir=OUT_DIR,
    split_samples=samples,
    image_root_rel=MILK_ROOT_REL,
)


## 5. Validate and preview


In [ ]:
preview_processed(OUT_DIR, "milk10k/clinical_context")
